# 🌍 PaleoPAL — Your Paleoclimate AI Research Assistant

**Two core capabilities:**
- **Ask PaleoPAL** — research Q&A powered by Claude
- **Analyze proxy data** — statistical analysis and visualization for ice cores, tree rings, and sediment/pollen records

---

## 1. Setup — Install dependencies & configure API

In [ ]:
# Install required packages (run once)
import subprocess, sys
packages = ["anthropic", "pandas", "numpy", "matplotlib", "scipy", "ipywidgets"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("All packages ready.")

In [ ]:
import os
import anthropic
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats, signal
from scipy.interpolate import interp1d
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
warnings.filterwarnings('ignore')

# ── API KEY ─────────────────────────────────────────────────────────────────
# Option A: paste your key directly (for local use only — never commit to git)
# ANTHROPIC_API_KEY = "sk-ant-..."

# Option B: set as environment variable (recommended)
# In your terminal: export ANTHROPIC_API_KEY="sk-ant-..."

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
if not api_key:
    api_key = input("Paste your Anthropic API key: ").strip()

client = anthropic.Anthropic(api_key=api_key)
print("PaleoPAL initialized. API key configured.")
# ── SYNTHETIC FALLBACK DATASETS (always available offline) ────────────────────
np.random.seed(42)

def _make_ice_core():
    age  = np.linspace(0, 50000, 500)
    d18O = (-2.5 * np.cos(2 * np.pi * age / 23000)
            - 1.8 * np.cos(2 * np.pi * age / 41000)
            + 0.5 * np.random.randn(500))
    return pd.DataFrame({"age": age, "d18O": d18O})

def _make_tree_rings():
    year = np.arange(1500, 2024)
    rw   = (1.0 - 0.0003 * (year - 1500)
            + 0.15 * np.sin(2 * np.pi * (year - 1500) / 11)
            + 0.08 * np.random.randn(len(year)))
    return pd.DataFrame({"age": year, "ring_width": np.clip(rw, 0.2, None)})

def _make_sediment():
    age    = np.linspace(0, 12000, 300)
    pollen = (35 + 15 * np.exp(-((age - 6000)**2) / (2 * 2000**2))
              + 5 * np.sin(2 * np.pi * age / 1500)
              + 3 * np.random.randn(300))
    return pd.DataFrame({"age": age, "pollen_pct": np.clip(pollen, 5, 80)})

SAMPLES = {
    "ice_core":   _make_ice_core(),
    "tree_rings": _make_tree_rings(),
    "sediment":   _make_sediment(),
}
print("Synthetic fallback datasets ready: ice_core, tree_rings, sediment")
print("(These are used when no real dataset is loaded — replace with fetch_noaa / fetch_neotoma / fetch_pangaea)")


---
## 2. PaleoPAL Research Q&A
Ask any paleoclimatology question. PaleoPAL acts as a specialist researcher with deep knowledge of proxy methods, climate history, and data interpretation.

In [ ]:
# ── SYSTEM PROMPT ────────────────────────────────────────────────────────────
PALEOPAL_SYSTEM = """
You are PaleoPAL, an expert AI research assistant specialising in paleoclimatology and 
paleoclimate data analysis. You have deep knowledge of:

PROXY METHODS:
- Ice cores: stable isotopes (δ18O, δD), dust layers, gas bubbles, accumulation rates
- Tree rings (dendrochronology): ring width, density, isotopic composition, MXD
- Sediment & pollen records: varves, foraminifera, pollen assemblages, biomarkers, grain size

ANALYTICAL METHODS:
- Age-depth modelling (Bacon, OxCal, COPRA)
- Spectral analysis (Lomb-Scargle, MTM, wavelet)
- Climate reconstruction techniques (RegEM, CPS, PCA)
- Uncertainty quantification and calibration

CLIMATE SCIENCE:
- Orbital forcing (Milankovitch cycles), ENSO, AMO, NAO teleconnections
- Holocene, Last Glacial Maximum, Dansgaard-Oeschger events
- Regional climate variability and global synthesis

DATABASES & RESOURCES:
- NOAA Paleoclimatology, Neotoma, PANGAEA, LiPD, PAGES 2k

Guidelines:
- Be precise and cite relevant concepts, methods, or studies where appropriate
- When interpreting data, note uncertainty and limitations
- Suggest next analytical steps when relevant
- Keep answers focused and scientifically rigorous
"""

conversation_history = []

def ask_paleopal(question: str, verbose: bool = True) -> str:
    """Send a question to PaleoPAL and return the answer. Maintains conversation history."""
    conversation_history.append({"role": "user", "content": question})
    
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1500,
        system=PALEOPAL_SYSTEM,
        messages=conversation_history
    )
    answer = response.content[0].text
    conversation_history.append({"role": "assistant", "content": answer})
    
    if verbose:
        display(Markdown(f"**PaleoPAL:** {answer}"))
    return answer

def reset_conversation():
    """Clear conversation history to start a fresh topic."""
    conversation_history.clear()
    print("Conversation reset.")

print("Q&A module ready. Use ask_paleopal('your question') or the interactive widget below.")

In [ ]:
# ── INTERACTIVE Q&A WIDGET ───────────────────────────────────────────────────
question_box = widgets.Textarea(
    placeholder="e.g. What can δ18O values from ice cores tell us about past temperatures?",
    layout=widgets.Layout(width="100%", height="80px")
)
ask_btn     = widgets.Button(description="Ask PaleoPAL", button_style="primary")
reset_btn   = widgets.Button(description="Reset conversation", button_style="warning")
output_area = widgets.Output()

def on_ask(b):
    q = question_box.value.strip()
    if not q:
        return
    with output_area:
        clear_output(wait=True)
        display(Markdown(f"**You:** {q}"))
        display(Markdown("*Thinking...*"))
        clear_output(wait=True)
        display(Markdown(f"**You:** {q}"))
        ask_paleopal(q)

def on_reset(b):
    reset_conversation()
    with output_area:
        clear_output()
        print("Conversation cleared.")

ask_btn.on_click(on_ask)
reset_btn.on_click(on_reset)

display(widgets.VBox([
    widgets.HTML("<h3>Ask PaleoPAL</h3>"),
    question_box,
    widgets.HBox([ask_btn, reset_btn]),
    output_area
]))

In [ ]:
# ── QUICK EXAMPLES (uncomment to try) ────────────────────────────────────────
# ask_paleopal("What are Dansgaard-Oeschger events and what do ice cores reveal about them?")
# ask_paleopal("How do I calibrate radiocarbon dates from a lake sediment core?")
# ask_paleopal("What spectral analysis method is best for unevenly spaced proxy time series?")

---
## 3. Real Proxy Data — Fetch from Public Databases

PaleoPAL downloads real paleoclimate data from three major public archives:

| Database | Best for | Function |
|---|---|---|
| **NOAA Paleoclimatology** | Ice cores, tree rings, corals | `search_noaa()` / `fetch_noaa()` |
| **Neotoma Paleoecology** | Pollen, lake sediments | `search_neotoma()` / `fetch_neotoma()` |
| **PANGAEA** | Marine & lake sediment cores | `search_pangaea()` / `fetch_pangaea()` |

All fetch functions return a `dataset` dict fully compatible with `analyze_proxy()`, `plot_proxy()`, and `interpret_proxy()`.

In [ ]:
# ── REAL DATA FETCHERS ────────────────────────────────────────────────────────
import requests, io

def _http_get(url, params=None, timeout=20):
    """HTTP GET with clear error messages."""
    try:
        r = requests.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        return r
    except requests.exceptions.ConnectionError:
        raise ConnectionError(f"Cannot reach {url}. Check your internet connection.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"HTTP {e.response.status_code} from {url}")

# ════════════════════════════════════════════════════════════════════════════
# A. NOAA PALEOCLIMATOLOGY  https://www.ncei.noaa.gov/products/paleoclimatology
# ════════════════════════════════════════════════════════════════════════════
NOAA_SEARCH = "https://www.ncei.noaa.gov/access/paleo-search/study/search.json"
NOAA_STUDY  = "https://www.ncei.noaa.gov/access/paleo-search/study/"
NOAA_DATA_TYPE = {
    "ice_core":   18,
    "tree_rings": 5,
    "sediment":   16,
    "coral":      9,
    "speleothem": 15,
}

def search_noaa(proxy_type="ice_core", keyword="", max_results=10):
    """
    Search NOAA Paleoclimatology for studies.

    Parameters
    ----------
    proxy_type  : 'ice_core', 'tree_rings', 'sediment', 'coral', or 'speleothem'
    keyword     : optional text filter (investigator name, site, or region)
    max_results : number of records to return

    Returns list of dicts with 'id', 'name', 'investigators', 'doi'.
    """
    r = _http_get(NOAA_SEARCH, params={
        "dataPublisher": "NOAA",
        "dataTypeId":    NOAA_DATA_TYPE.get(proxy_type, 18),
        "keywords":      keyword,
        "pageSize":      max_results,
        "page":          1,
    })
    studies = r.json().get("studyList", [])
    return [{"id": s.get("xmlId",""), "name": s.get("studyName",""),
             "investigators": s.get("investigators",""),
             "min_year": s.get("earliestYearBP"), "doi": s.get("doi","")}
            for s in studies]


def fetch_noaa(study_id, proxy_type="ice_core", age_col=None, data_col=None):
    """
    Fetch a NOAA Paleoclimatology study by its xmlId.

    Parameters
    ----------
    study_id   : str — NOAA xmlId from search_noaa() (e.g. '2475' for GISP2)
    proxy_type : str — proxy label used downstream
    age_col    : str — age column name (auto-detected from headers if None)
    data_col   : str — proxy column name (auto-detected if None)

    How it works
    ------------
    1. Fetches study metadata JSON to get the list of linked data file URLs
    2. Downloads the first .txt file found
    3. Skips header comment lines (those starting with #)
    4. Parses the tab-delimited data into a DataFrame
    """
    meta = _http_get(f"{NOAA_STUDY}{study_id}.json").json()
    file_url = None
    for f in meta.get("onlineResourceList", []):
        url = f.get("url", "")
        if url.endswith(".txt") or "data" in url.lower():
            file_url = url
            break
    if not file_url:
        raise ValueError(
            f"No data file found for study {study_id}.\n"
            f"Browse manually: https://www.ncei.noaa.gov/access/paleo-search/study/{study_id}"
        )
    raw   = _http_get(file_url).text
    lines = [l for l in raw.splitlines() if not l.strip().startswith("#") and l.strip()]
    df    = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", on_bad_lines="skip")
    df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
    print(f"NOAA study {study_id} fetched — {len(df)} rows, columns: {list(df.columns[:6])}")
    return load_proxy_data(df, proxy_type, age_col, data_col)


# ════════════════════════════════════════════════════════════════════════════
# B. NEOTOMA PALEOECOLOGY DATABASE  https://api.neotomadb.org/
# ════════════════════════════════════════════════════════════════════════════
NEOTOMA_BASE = "https://api.neotomadb.org/v2.0"

def search_neotoma(taxon="Pinus", dataset_type="pollen", limit=10):
    """
    Search Neotoma for sites containing a given pollen taxon.

    Parameters
    ----------
    taxon        : pollen taxon (e.g. 'Quercus', 'Betula', 'Picea')
    dataset_type : 'pollen', 'vertebrate fauna', 'plant macrofossils'
    limit        : max number of results

    Returns list of dicts with 'dataset_id', 'site_name', 'lat', 'lon'.
    """
    r = _http_get(f"{NEOTOMA_BASE}/data/datasets",
                 params={"taxonname": taxon, "datasettype": dataset_type, "limit": limit})
    items = r.json().get("data", [])
    results = []
    for item in items:
        site  = item.get("site", {})
        coords = site.get("geography", {}).get("coordinates", [None, None])
        results.append({
            "dataset_id":   item.get("datasetid"),
            "site_name":    site.get("sitename", ""),
            "lat":          coords[1] if len(coords) > 1 else None,
            "lon":          coords[0] if coords else None,
            "dataset_type": item.get("datasettype", ""),
        })
    return results


def fetch_neotoma(dataset_id, taxon=None):
    """
    Fetch pollen samples from a Neotoma dataset.

    Parameters
    ----------
    dataset_id : int — from search_neotoma() results
    taxon      : str — if given, returns % abundance of this taxon;
                       if None, returns total pollen sum per sample

    How it works
    ------------
    Calls the Neotoma /samples endpoint, extracts calibrated ages (cal yr BP)
    and pollen counts, computes percentage abundance, returns a sediment dataset.
    """
    r    = _http_get(f"{NEOTOMA_BASE}/data/datasets/{dataset_id}/samples")
    data = r.json().get("data", [])
    records = []
    for sample in data:
        age = sample.get("age")
        if age is None:
            continue
        counts = {d["taxonname"]: d["value"]
                  for d in sample.get("data", []) if d.get("elementtype") == "pollen"}
        total = sum(counts.values()) or 1
        pct   = counts.get(taxon, 0) / total * 100 if taxon else float(total)
        records.append({"age": float(age), "pollen_pct": pct})

    if not records:
        raise ValueError(f"No pollen data found for Neotoma dataset {dataset_id}.")

    df    = pd.DataFrame(records).sort_values("age").reset_index(drop=True)
    label = f"{taxon} %" if taxon else "Total pollen sum"
    print(f"Neotoma dataset {dataset_id} — {len(df)} samples  |  variable: {label}")
    return {"df": df, "proxy_type": "sediment", "age_col": "age", "data_col": "pollen_pct"}


# ════════════════════════════════════════════════════════════════════════════
# C. PANGAEA  https://www.pangaea.de
# ════════════════════════════════════════════════════════════════════════════
def search_pangaea(keyword, max_results=10):
    """
    Search PANGAEA for datasets by keyword.

    Parameters
    ----------
    keyword     : free-text search (e.g. 'Holocene lake pollen', 'ice core isotope')
    max_results : number of results to return

    Returns list of dicts with 'pangaea_id', 'title', 'authors', 'doi'.
    """
    r = _http_get("https://www.pangaea.de/api/datasets/search",
                  params={"q": keyword, "count": max_results})
    return [{"pangaea_id": item.get("URI","").split("/")[-1],
             "title":       item.get("title","")[:80],
             "authors":     ", ".join(a.get("lastName","") for a in item.get("authors",[])),
             "doi":         item.get("URI","")}
            for item in r.json().get("results", [])]


def fetch_pangaea(pangaea_id, proxy_type="sediment", age_col=None, data_col=None):
    """
    Fetch a PANGAEA dataset by its numeric ID.

    Parameters
    ----------
    pangaea_id : str or int — e.g. '728846' or '10.1594/PANGAEA.728846'
    proxy_type : proxy label for downstream analysis
    age_col    : age column name (auto-detected if None)
    data_col   : proxy column name (auto-detected if None)

    How it works
    ------------
    PANGAEA serves tab-delimited data via a direct ?format=textfile URL.
    Header comments are enclosed in /* ... */ blocks and stripped before parsing.
    """
    pid = str(pangaea_id).replace("10.1594/PANGAEA.", "")
    raw = _http_get(f"https://doi.pangaea.de/10.1594/PANGAEA.{pid}?format=textfile").text
    lines, skip = [], False
    for line in raw.splitlines():
        if line.startswith("/*"):  skip = True
        if line.startswith("*/"):  skip = False; continue
        if not skip and line.strip():
            lines.append(line)
    df = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", on_bad_lines="skip")
    df.columns = [c.strip().lower().replace(" ","_").replace("/","_per_") for c in df.columns]
    print(f"PANGAEA {pid} — {len(df)} rows, columns: {list(df.columns[:6])}")
    return load_proxy_data(df, proxy_type, age_col, data_col)




print("Real-data fetchers ready:")
print("  NOAA    — search_noaa(proxy_type, keyword)   /  fetch_noaa(study_id)")
print("  Neotoma — search_neotoma(taxon)              /  fetch_neotoma(dataset_id, taxon)")
print("  PANGAEA — search_pangaea(keyword)            /  fetch_pangaea(pangaea_id)")


In [ ]:
# ── CURATED REAL DATASETS — verified IDs ready to use ────────────────────────
# Un-comment ONE block, run this cell, then run the full pipeline in Section 4.

# ── ICE CORES (NOAA) ─────────────────────────────────────────────────────────
# GISP2 Greenland Ice Core δ18O — 110,000 yr record (Grootes & Stuiver 1997)
# dataset = fetch_noaa("2475", proxy_type="ice_core")

# Vostok Antarctica δD — 420,000 yr record (Petit et al. 1999)
# dataset = fetch_noaa("2437", proxy_type="ice_core")

# EPICA Dome C δD — 800,000 yr record (Jouzel et al. 2007)
# dataset = fetch_noaa("15076", proxy_type="ice_core")

# ── TREE RINGS (NOAA ITRDB) ───────────────────────────────────────────────────
# Search for chronologies first, then fetch the one you want:
# studies = search_noaa("tree_rings", keyword="bristlecone", max_results=8)
# for s in studies:
#     print(s["id"], "|", s["name"][:60])
# dataset = fetch_noaa(studies[0]["id"], proxy_type="tree_rings")

# ── POLLEN RECORDS (NEOTOMA) ──────────────────────────────────────────────────
# Search first to find a site that interests you:
# sites = search_neotoma(taxon="Quercus", dataset_type="pollen", limit=8)
# for s in sites:
#     print(s["dataset_id"], "|", s["site_name"], "| lat:", s["lat"])
# Then fetch:
# dataset = fetch_neotoma(dataset_id=4116, taxon="Quercus")  # Oak % pollen

# ── SEDIMENT CORES (PANGAEA) ──────────────────────────────────────────────────
# Search first:
# results = search_pangaea("Holocene lake sediment pollen Europe", max_results=8)
# for r in results:
#     print(r["pangaea_id"], "|", r["title"])
# Then fetch:
# dataset = fetch_pangaea("728846", proxy_type="sediment")

# ── WORKFLOW ──────────────────────────────────────────────────────────────────
# Once you have fetched a real dataset, pass it through the full pipeline:
#
#   results = analyze_proxy(dataset)
#   plot_proxy(results)
#   plot_spectral(results)
#   interpret_proxy(results)

print("Curated dataset cell ready — uncomment a block above and run Section 4.")


In [ ]:
# ── DATA LOADER ───────────────────────────────────────────────────────────────
def load_proxy_data(source, proxy_type=None, age_col=None, data_col=None):
    """
    Load proxy data. Accepts:
      - 'ice_core', 'tree_rings', or 'sediment'  → synthetic fallback sample
      - a file path string                        → reads CSV from disk
      - a pandas DataFrame                        → uses directly

    age_col / data_col are auto-detected from column names if not specified.
    """
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, str) and source in SAMPLES:
        df         = SAMPLES[source].copy()
        proxy_type = proxy_type or source
    elif isinstance(source, str):
        df = pd.read_csv(source)
        print(f"Loaded '{source}': {len(df)} rows, columns: {list(df.columns)}")
    else:
        raise ValueError(f"source must be a DataFrame, a sample key, or a CSV path. Got: {type(source)}")

    # Normalise column names
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # Auto-detect age column
    if age_col is None:
        candidates = [c for c in df.columns
                      if any(k in c.lower() for k in ["age","year","yr","time","depth","ka"])]
        age_col = candidates[0] if candidates else df.columns[0]

    # Auto-detect data column
    if data_col is None:
        data_col = next((c for c in df.columns if c != age_col), df.columns[1])

    # Auto-detect proxy type from column name
    if proxy_type is None:
        col_lower = data_col.lower()
        if any(k in col_lower for k in ["d18o","dd","deuterium","dust","accum"]):
            proxy_type = "ice_core"
        elif any(k in col_lower for k in ["ring","width","density","chronology"]):
            proxy_type = "tree_rings"
        else:
            proxy_type = "sediment"

    df2 = (df[[age_col, data_col]]
           .apply(pd.to_numeric, errors="coerce")
           .dropna()
           .sort_values(age_col)
           .reset_index(drop=True))

    print(f"Proxy type : {proxy_type}")
    print(f"Age column : '{age_col}'   Data column: '{data_col}'   N = {len(df2)}")
    return {"df": df2, "proxy_type": proxy_type, "age_col": age_col, "data_col": data_col}

print("load_proxy_data() ready.")


In [ ]:
# ── ANALYSIS ENGINE ───────────────────────────────────────────────────────────
def analyze_proxy(dataset: dict, window: int = 50) -> dict:
    """
    Run a full statistical analysis on a proxy dataset.
    Returns a results dict consumed by plot_proxy() and interpret_proxy().
    """
    df       = dataset["df"]
    age_col  = dataset["age_col"]
    data_col = dataset["data_col"]
    ptype    = dataset["proxy_type"]
    x        = df[age_col].values
    y        = df[data_col].values

    # ── Basic stats ──────────────────────────────────────────────────────────
    stats_dict = {
        "n":        len(y),
        "mean":     float(np.mean(y)),
        "std":      float(np.std(y)),
        "min":      float(np.min(y)),
        "max":      float(np.max(y)),
        "range":    float(np.max(y) - np.min(y)),
        "skewness": float(stats.skew(y)),
        "kurtosis": float(stats.kurtosis(y)),
        "age_span": (float(np.min(x)), float(np.max(x))),
    }

    # ── Linear trend ─────────────────────────────────────────────────────────
    slope, intercept, r, p_val, _ = stats.linregress(x, y)
    trend = {"slope": slope, "intercept": intercept, "r": r, "p": p_val,
             "trend_line": slope * x + intercept}

    # ── Rolling statistics ────────────────────────────────────────────────────
    series      = pd.Series(y)
    roll_mean   = series.rolling(window, center=True, min_periods=1).mean().values
    roll_std    = series.rolling(window, center=True, min_periods=1).std().values

    # ── Anomalies (z-score) ───────────────────────────────────────────────────
    anomaly = (y - np.mean(y)) / np.std(y)

    # ── Spectral analysis (Lomb-Scargle for unevenly spaced data) ────────────
    age_range   = np.max(x) - np.min(x)
    freq_min    = 1 / (age_range * 0.5)
    freq_max    = 1 / (np.median(np.diff(np.sort(x))) * 2) if len(x) > 2 else 1
    freqs       = np.linspace(freq_min, freq_max, 512)
    power       = signal.lombscargle(x, y - np.mean(y), 2 * np.pi * freqs, normalize=True)
    periods     = 1 / freqs
    top_idx     = np.argsort(power)[-5:][::-1]
    top_periods = periods[top_idx]
    top_powers  = power[top_idx]

    # ── Proxy-specific metrics ────────────────────────────────────────────────
    proxy_metrics = {}
    if ptype == "ice_core":
        # Estimate temperature from δ18O (simplified 0.5‰ per °C calibration)
        proxy_metrics["approx_temp_range_C"] = stats_dict["range"] / 0.5
        proxy_metrics["glacial_interglacial"] = "possible" if stats_dict["range"] > 3 else "subdued"
    elif ptype == "tree_rings":
        # Expressed Population Signal (simplified)
        proxy_metrics["cv"] = stats_dict["std"] / stats_dict["mean"] if stats_dict["mean"] != 0 else np.nan
        # First-order autocorrelation
        proxy_metrics["autocorr_lag1"] = float(pd.Series(y).autocorr(lag=1))
        # Percent above/below mean (high/low growth years)
        proxy_metrics["pct_above_mean"] = float(np.mean(y > np.mean(y)) * 100)
    elif ptype == "sediment":
        proxy_metrics["holocene_trend"] = "warming" if slope < 0 else "cooling"  # older = higher BP
        proxy_metrics["variability_pct"] = float(stats_dict["std"] / abs(stats_dict["mean"]) * 100)

    return {
        "x": x, "y": y,
        "age_col": age_col, "data_col": data_col, "proxy_type": ptype,
        "stats": stats_dict,
        "trend": trend,
        "roll_mean": roll_mean, "roll_std": roll_std,
        "anomaly": anomaly,
        "freqs": freqs, "periods": periods, "power": power,
        "top_periods": top_periods, "top_powers": top_powers,
        "proxy_metrics": proxy_metrics,
        "window": window,
    }

print("analyze_proxy() ready.")

In [ ]:
# ── VISUALIZATION ENGINE ──────────────────────────────────────────────────────
PROXY_LABELS = {
    "ice_core":   {"color": "#378ADD", "ylabel": "δ¹⁸O (‰ VSMOW)",   "title": "Ice Core Record"},
    "tree_rings": {"color": "#639922", "ylabel": "Ring Width Index",   "title": "Tree-Ring Chronology"},
    "sediment":   {"color": "#BA7517", "ylabel": "Pollen %",           "title": "Sediment / Pollen Record"},
}

def plot_proxy(results: dict):
    """Generate a 4-panel analysis figure for a proxy dataset."""
    ptype  = results["proxy_type"]
    meta   = PROXY_LABELS.get(ptype, {"color": "#534AB7", "ylabel": results["data_col"], "title": "Proxy Record"})
    color  = meta["color"]
    x, y   = results["x"], results["y"]
    xlabel = results["age_col"].replace("_", " ").title()

    fig = plt.figure(figsize=(14, 10))
    fig.suptitle(f"PaleoPAL Analysis — {meta['title']}", fontsize=14, fontweight="bold", y=0.98)
    gs  = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.35)

    # ── Panel 1: Raw time series + smoothing ──────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(x, y, color=color, alpha=0.35, linewidth=0.8, label="Raw data")
    ax1.plot(x, results["roll_mean"], color=color, linewidth=2.0,
             label=f"Rolling mean (n={results['window']})")
    ax1.fill_between(x,
                     results["roll_mean"] - results["roll_std"],
                     results["roll_mean"] + results["roll_std"],
                     color=color, alpha=0.12, label="±1 SD envelope")
    ax1.plot(x, results["trend"]["trend_line"], "--", color="#E24B4A",
             linewidth=1.4, label=f"Trend (r={results['trend']['r']:.2f}, p={results['trend']['p']:.3f})")
    ax1.set_xlabel(xlabel); ax1.set_ylabel(meta["ylabel"])
    ax1.set_title("Time Series with Smoothing & Trend")
    ax1.legend(fontsize=8, loc="best"); ax1.grid(True, alpha=0.25)

    # ── Panel 2: Anomaly / z-score ────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    pos = results["anomaly"] >= 0
    ax2.bar(x[pos],  results["anomaly"][pos],  color=color,    alpha=0.7, width=(x[-1]-x[0])/len(x)*0.9)
    ax2.bar(x[~pos], results["anomaly"][~pos], color="#E24B4A", alpha=0.7, width=(x[-1]-x[0])/len(x)*0.9)
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.axhline(2, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
    ax2.axhline(-2, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
    ax2.set_xlabel(xlabel); ax2.set_ylabel("Z-score")
    ax2.set_title("Anomaly (Z-score)"); ax2.grid(True, alpha=0.20)

    # ── Panel 3: Distribution ─────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.hist(y, bins=30, color=color, alpha=0.70, edgecolor="white", linewidth=0.5)
    mu, sigma = results["stats"]["mean"], results["stats"]["std"]
    xfit = np.linspace(y.min(), y.max(), 200)
    yfit = stats.norm.pdf(xfit, mu, sigma) * len(y) * (y.max() - y.min()) / 30
    ax3.plot(xfit, yfit, "--", color="#E24B4A", linewidth=1.8, label="Normal fit")
    ax3.axvline(mu, color="black", linewidth=1.2, label=f"Mean = {mu:.2f}")
    ax3.set_xlabel(meta["ylabel"]); ax3.set_ylabel("Count")
    ax3.set_title("Distribution"); ax3.legend(fontsize=8); ax3.grid(True, alpha=0.20)

    plt.savefig("paleoPAL_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved as paleoPAL_analysis.png")


def plot_spectral(results: dict):
    """Plot Lomb-Scargle periodogram with annotated dominant periods."""
    ptype = results["proxy_type"]
    color = PROXY_LABELS.get(ptype, {"color": "#534AB7"})["color"]

    # Reference periods by proxy type
    ref_lines = {
        "ice_core":   [(23000, "Precession"), (41000, "Obliquity"), (100000, "Eccentricity")],
        "tree_rings": [(11, "Solar ~11yr"), (22, "Hale ~22yr")],
        "sediment":   [(1500, "Bond cycle"), (2300, "Hallstatt")],
    }

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(results["periods"], results["power"], color=color, linewidth=1.2)
    ax.fill_between(results["periods"], results["power"], alpha=0.15, color=color)

    for p, lbl in ref_lines.get(ptype, []):
        prange = (results["periods"].min(), results["periods"].max())
        if prange[0] <= p <= prange[1]:
            ax.axvline(p, color="#E24B4A", linewidth=1.0, linestyle="--", alpha=0.7)
            ax.text(p, ax.get_ylim()[1] * 0.85, lbl, rotation=90,
                    fontsize=7, color="#E24B4A", ha="right", va="top")

    for period in results["top_periods"][:3]:
        ax.axvline(period, color="#1D9E75", linewidth=0.8, linestyle=":", alpha=0.8)

    ax.set_xlabel("Period (years)"); ax.set_ylabel("Normalised power")
    ax.set_title("Lomb-Scargle Periodogram — Dominant Periodicities")
    ax.grid(True, alpha=0.20); ax.set_xscale("log")
    plt.tight_layout()
    plt.savefig("paleoPAL_spectral.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Spectral figure saved as paleoPAL_spectral.png")

print("Visualization functions ready.")

In [ ]:
# ── AI INTERPRETATION ─────────────────────────────────────────────────────────
def interpret_proxy(results: dict) -> str:
    """
    Ask PaleoPAL to interpret the statistical results in scientific context.
    Returns the interpretation string and prints it.
    """
    s      = results["stats"]
    t      = results["trend"]
    ptype  = results["proxy_type"]
    dcol   = results["data_col"]
    top_p  = ", ".join([f"{p:.0f} yr" for p in results["top_periods"][:3]])
    pm     = results["proxy_metrics"]

    prompt = f"""
I have analysed a {ptype.replace('_', ' ')} proxy dataset. Please provide a concise 
scientific interpretation of these results, noting what they may indicate about past 
climate, any caveats, and suggested next steps.

VARIABLE: {dcol}
RECORD LENGTH: {s['age_span'][0]:.0f} – {s['age_span'][1]:.0f} (age units)
N SAMPLES: {s['n']}

DESCRIPTIVE STATISTICS:
  Mean: {s['mean']:.3f}, Std: {s['std']:.3f}, Range: {s['range']:.3f}
  Skewness: {s['skewness']:.2f}, Kurtosis: {s['kurtosis']:.2f}

LINEAR TREND:
  Slope: {t['slope']:.6f} per unit age, r = {t['r']:.3f}, p = {t['p']:.4f}

DOMINANT PERIODS (Lomb-Scargle): {top_p}

PROXY-SPECIFIC METRICS: {pm}

Please structure your response with:
1. Overall climate signal
2. Notable features (trend, variability, periodicities)
3. Caveats and uncertainties
4. Recommended next analytical steps
"""
    display(Markdown("### PaleoPAL Interpretation"))
    return ask_paleopal(prompt)

print("interpret_proxy() ready.")

---
## 4. Run a Full Analysis
Choose a sample dataset or load your own CSV. The cell below runs the complete pipeline.

In [ ]:
# ── FULL PIPELINE ─────────────────────────────────────────────────────────────
# Change 'ice_core' to 'tree_rings', 'sediment', or a path to your own CSV file.

PROXY_SOURCE = "ice_core"   # ← edit this
# PROXY_SOURCE = "tree_rings"
# PROXY_SOURCE = "sediment"
# PROXY_SOURCE = "my_data.csv"  # your own file

# 1. Load
dataset = load_proxy_data(PROXY_SOURCE)

# 2. Analyse
results = analyze_proxy(dataset, window=30)

# 3. Print summary statistics
s = results["stats"]
print(f"\n{'─'*50}")
print(f"  Summary: {results['data_col']}")
print(f"{'─'*50}")
print(f"  N = {s['n']}  |  Age span: {s['age_span'][0]:.0f} – {s['age_span'][1]:.0f}")
print(f"  Mean: {s['mean']:.3f}  |  Std: {s['std']:.3f}  |  Range: {s['range']:.3f}")
print(f"  Skew: {s['skewness']:.2f}  |  Kurtosis: {s['kurtosis']:.2f}")
t = results["trend"]
print(f"  Trend slope: {t['slope']:.2e}  |  r = {t['r']:.3f}  |  p = {t['p']:.4f}")
print(f"  Top periods: {', '.join([f'{p:.0f} yr' for p in results['top_periods'][:3]])}")
if results["proxy_metrics"]:
    print(f"  Proxy metrics: {results['proxy_metrics']}")
print(f"{'─'*50}\n")

# 4. Plot
plot_proxy(results)
plot_spectral(results)

# 5. AI interpretation
interpret_proxy(results)

---
## 5. Compare Two Proxy Records
Useful for teleconnection analysis, proxy replication, or comparing different archives.

In [ ]:
def compare_proxies(source_a, source_b, label_a=None, label_b=None):
    """
    Load two proxy datasets, interpolate to a common age grid,
    compute correlation, and ask PaleoPAL to interpret the comparison.
    """
    A = load_proxy_data(source_a)
    B = load_proxy_data(source_b)
    rA = analyze_proxy(A)
    rB = analyze_proxy(B)

    label_a = label_a or source_a
    label_b = label_b or source_b

    # Interpolate to common grid
    age_min = max(rA["x"].min(), rB["x"].min())
    age_max = min(rA["x"].max(), rB["x"].max())
    if age_min >= age_max:
        print("No overlapping age range between the two records.")
        return

    n_pts  = 200
    common = np.linspace(age_min, age_max, n_pts)
    ya     = interp1d(rA["x"], rA["y"], kind="linear", fill_value="extrapolate")(common)
    yb     = interp1d(rB["x"], rB["y"], kind="linear", fill_value="extrapolate")(common)

    r_val, p_val = stats.pearsonr(ya, yb)

    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    ca = PROXY_LABELS.get(rA["proxy_type"], {"color": "#534AB7"})["color"]
    cb = PROXY_LABELS.get(rB["proxy_type"], {"color": "#D85A30"})["color"]

    ax1.plot(common, ya, color=ca, linewidth=1.5, label=label_a)
    ax1.set_ylabel(rA["data_col"]); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.25)
    ax1.set_title(f"Proxy Comparison  |  r = {r_val:.3f}, p = {p_val:.4f}")

    ax2.plot(common, yb, color=cb, linewidth=1.5, label=label_b)
    ax2.set_xlabel(rA["age_col"]); ax2.set_ylabel(rB["data_col"])
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.savefig("paleoPAL_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nPearson r = {r_val:.3f}  |  p = {p_val:.4f}  |  Overlap: {age_min:.0f}–{age_max:.0f}")

    # AI interpretation
    prompt = f"""
I compared two paleoclimate proxy records over their overlapping time period:
  Record A: {label_a} ({rA['proxy_type'].replace('_',' ')}), variable: {rA['data_col']}
  Record B: {label_b} ({rB['proxy_type'].replace('_',' ')}), variable: {rB['data_col']}
  Overlapping age range: {age_min:.0f} – {age_max:.0f}
  Pearson correlation: r = {r_val:.3f}, p = {p_val:.4f}

What does this correlation suggest climatically? What caveats apply when comparing
these two proxy types? What further analysis would be valuable?
"""
    display(Markdown("### PaleoPAL: Comparison Interpretation"))
    ask_paleopal(prompt)

# ── Run comparison (edit sources as needed) ───────────────────────────────────
compare_proxies("ice_core", "sediment", label_a="Ice Core δ¹⁸O", label_b="Sediment Pollen %")

---
## 6. Quick Reference

| Function | What it does |
|---|---|
| `ask_paleopal(question)` | Ask any research question (multi-turn) |
| `reset_conversation()` | Clear conversation history |
| `load_proxy_data(source)` | Load CSV or sample dataset |
| `analyze_proxy(dataset)` | Full statistical analysis |
| `plot_proxy(results)` | 3-panel time series figure |
| `plot_spectral(results)` | Lomb-Scargle periodogram |
| `interpret_proxy(results)` | AI scientific interpretation |
| `compare_proxies(a, b)` | Correlate and compare two records |

**Supported sample datasets:** `'ice_core'`, `'tree_rings'`, `'sediment'`  
**Own data:** pass a CSV path — columns are auto-detected, or specify `age_col` / `data_col`.